# Baumkataster Analysis

Explores the Magdeburg city tree cadastre 2026 (`Baeume_SFM_2026.gpkg`):
species distribution, crown geometry, and allometric calibration of the height model.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import box

from src.config import AREAS, ALLOMETRIC_A, ALLOMETRIC_B, OUTPUT_DIR

## Load Baumkataster

In [ ]:
BAUMKATASTER_PATH = Path("../references/Baeume_SFM_2026.gpkg")
bk = gpd.read_file(BAUMKATASTER_PATH)  # already EPSG:25832

# Columns:
#   Gattung lang        — species (Latin + German name)
#   Baumhoehe           — total height (m)
#   Kronendurchmesser   — crown diameter (m)
#   Stammumfang         — trunk circumference (cm)
#   Pflanzjahr          — planting year
#   Objektart lang      — location type (Öffentliches Grün / AMT 66 / Spielplatz)

print(f"Total trees (Magdeburg): {len(bk):,}")
print(f"CRS: {bk.crs}")
bk[["Gattung lang", "Baumhoehe", "Kronendurchmesser", "Stammumfang", "Pflanzjahr"]].describe()

## Species and location breakdown

In [ ]:
# Top species — citywide
print("=== TOP 20 SPECIES (Magdeburg citywide) ===")
print(bk["Gattung lang"].value_counts().head(20).to_string())

print()
print("=== LOCATION TYPE ===")
print(bk["Objektart lang"].value_counts().to_string())

## OVGU study area — crown diameter map

In [ ]:
from pyproj import Transformer

_to_utm = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True)
ovgu = AREAS["ovgu_bbox"]
west_m, south_m = _to_utm.transform(ovgu["west"], ovgu["south"])
east_m, north_m = _to_utm.transform(ovgu["east"], ovgu["north"])
bbox_utm = gpd.GeoDataFrame(
    geometry=[box(west_m, south_m, east_m, north_m)],
    crs="EPSG:25832",
)

bk_ovgu = bk.clip(bbox_utm).copy()
bk_ovgu = bk_ovgu[bk_ovgu["Kronendurchmesser"] > 0].reset_index(drop=True)

print(f"Trees in OVGU bbox (with crown data): {len(bk_ovgu)}")
print(f"Crown diameter — mean: {bk_ovgu['Kronendurchmesser'].mean():.1f} m  "
      f"median: {bk_ovgu['Kronendurchmesser'].median():.1f} m  "
      f"max: {bk_ovgu['Kronendurchmesser'].max():.1f} m")

print("\n--- Species (OVGU) ---")
print(bk_ovgu["Gattung lang"].value_counts().head(15).to_string())

# Buffer each point by crown radius → actual circular footprint in map units
bk_crown = bk_ovgu.copy()
bk_crown["geometry"] = bk_crown.apply(
    lambda r: r.geometry.buffer(r["Kronendurchmesser"] / 2), axis=1
)

norm = mcolors.Normalize(vmin=bk_ovgu["Kronendurchmesser"].quantile(0.05),
                         vmax=bk_ovgu["Kronendurchmesser"].quantile(0.95))
cmap = cm.YlGn
colors = cmap(norm(bk_crown["Kronendurchmesser"].values))

fig, ax = plt.subplots(figsize=(10, 10))
bk_crown.plot(ax=ax, color=colors, edgecolor="darkgreen", linewidth=0.3, alpha=0.7)
bbox_utm.boundary.plot(ax=ax, color="red", linewidth=1.5, label="OVGU bbox")

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Crown diameter (m)", shrink=0.6)

ax.set_title(f"Baumkataster — OVGU area ({len(bk_crown)} trees, crown circles to scale)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
plt.tight_layout()
plt.show()

## Majority species per tile (250 m grid)

In [ ]:
from src.data_preprocessing import tiles_for_area
from src.config import TILE_SIZE_M
import matplotlib.patches as mpatches

# Build UTM tile polygons and find majority species in each
tiles = tiles_for_area(AREAS["ovgu_bbox"], TILE_SIZE_M)

records = []
for t in tiles:
    w_m, s_m = _to_utm.transform(t["west"], t["south"])
    e_m, n_m = _to_utm.transform(t["east"], t["north"])
    tile_poly = box(w_m, s_m, e_m, n_m)
    clipped = bk.clip(gpd.GeoDataFrame(geometry=[tile_poly], crs="EPSG:25832"))
    if len(clipped) == 0:
        majority, majority_n, majority_pct = "—", 0, 0.0
    else:
        vc = clipped["Gattung lang"].value_counts()
        majority = vc.index[0]
        majority_n = int(vc.iloc[0])
        majority_pct = majority_n / len(clipped) * 100
    records.append({
        "ix": t["ix"], "iy": t["iy"],
        "n_trees": len(clipped),
        "majority_species": majority,
        "majority_n": majority_n,
        "majority_pct": majority_pct,
        "geometry": tile_poly,
    })

tile_gdf = gpd.GeoDataFrame(records, crs="EPSG:25832")

# Print summary table
print(f"Tile size: {TILE_SIZE_M} m  |  Tiles: {len(tile_gdf)}\n")
for _, row in tile_gdf.sort_values(["iy", "ix"], ascending=[False, True]).iterrows():
    genus = row["majority_species"].split(",")[0]
    print(f"  tile ({row['ix']},{row['iy']})  n={row['n_trees']:>4}  "
          f"majority: {genus} ({row['majority_n']}, {row['majority_pct']:.0f}%)")

# Categorical color per majority species
species_list = tile_gdf["majority_species"].unique().tolist()
palette = plt.cm.Set2.colors
species_color = {s: palette[i % len(palette)] for i, s in enumerate(species_list)}

fig, ax = plt.subplots(figsize=(10, 10))

# Crown circles dimmed to grey-green background
bk_crown.plot(ax=ax, color="#aacfaa", edgecolor="none", alpha=0.4)

# Tile fill + boundary colored by majority species
for _, row in tile_gdf.iterrows():
    color = species_color[row["majority_species"]]
    gpd.GeoDataFrame([row], crs="EPSG:25832").plot(
        ax=ax, facecolor=color, edgecolor=color, linewidth=2.5, alpha=0.25
    )
    gpd.GeoDataFrame([row], crs="EPSG:25832").boundary.plot(
        ax=ax, color=color, linewidth=2.5
    )
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    genus = row["majority_species"].split(",")[0]
    ax.text(cx, cy, f"{genus}\n{row['majority_pct']:.0f}% of {row['n_trees']}",
            ha="center", va="center", fontsize=8,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2))

legend_patches = [
    mpatches.Patch(color=species_color[s], label=s.split(",")[0])
    for s in species_list
]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8, framealpha=0.9,
          title="Majority species")

ax.set_title(f"Majority Baumkataster species per {TILE_SIZE_M} m tile — OVGU area")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

## Allometric calibration from Baumkataster

In [ ]:
# Fit ln(H) = A + B·ln(CPA) on Baumkataster (citywide, valid measurements only)
cal = bk[["Baumhoehe", "Kronendurchmesser"]].copy()
cal = cal[(cal["Baumhoehe"] > 1) & (cal["Kronendurchmesser"] > 0.5)]  # drop zeros/tiny
cal["CPA_m2"] = math.pi * (cal["Kronendurchmesser"] / 2) ** 2
cal["ln_H"]   = np.log(cal["Baumhoehe"])
cal["ln_CPA"] = np.log(cal["CPA_m2"])

# OLS: ln(H) = A + B·ln(CPA)
X = np.column_stack([np.ones(len(cal)), cal["ln_CPA"].values])
B_hat, resid, _, _ = np.linalg.lstsq(X, cal["ln_H"].values, rcond=None)
A_urban, B_urban = B_hat

r2 = 1 - np.sum((cal["ln_H"] - (A_urban + B_urban * cal["ln_CPA"])) ** 2) / \
         np.sum((cal["ln_H"] - cal["ln_H"].mean()) ** 2)

print("=== Baumkataster-calibrated allometric model (citywide) ===")
print(f"  H = exp({A_urban:.3f} + {B_urban:.3f} · ln(CPA))   R² = {r2:.3f}   n = {len(cal):,}")
print()
print("=== Current config (src/config.py) ===")
print(f"  H = exp({ALLOMETRIC_A:.3f} + {ALLOMETRIC_B:.3f} · ln(CPA))")
print()

# Compare predictions at representative crown diameters
print(f"{'Crown D (m)':>12}  {'CPA (m²)':>10}  {'H_urban (m)':>12}  {'H_config (m)':>13}  {'Bk median H':>12}")
for cd in [4, 6, 8, 10, 12, 14]:
    cpa = math.pi * (cd / 2) ** 2
    h_urban = math.exp(A_urban + B_urban * math.log(cpa))
    h_config = math.exp(ALLOMETRIC_A + ALLOMETRIC_B * math.log(cpa))
    bk_med = bk[(bk["Kronendurchmesser"].between(cd - 0.5, cd + 0.5)) &
                (bk["Baumhoehe"] > 1)]["Baumhoehe"].median()
    print(f"{cd:>12}  {cpa:>10.1f}  {h_urban:>12.1f}  {h_config:>13.1f}  {bk_med:>12.1f}")

## Pipeline validation — height estimates vs Baumkataster measurements

In [ ]:
TILE_SIZE_VAL = 250
VMODEL        = "tcd_segformer"

trees_path = OUTPUT_DIR / "segments" / f"{TILE_SIZE_VAL}m" / \
             f"ovgu_bbox_{VMODEL}_trees_merged.fgb"
tree_gdf = gpd.read_file(trees_path)   # EPSG:25832

# Buffer BK points by crown radius → crown circles for spatial matching
bk_circles = bk[(bk["Kronendurchmesser"] > 0) & (bk["Baumhoehe"] > 1)].copy()
bk_circles["geometry"] = bk_circles.apply(
    lambda r: r.geometry.buffer(r["Kronendurchmesser"] / 2), axis=1
)

# Inner sjoin: pipeline polygon intersects BK circle
joined = gpd.sjoin(
    tree_gdf[["tree_id", "height_m", "crown_area_m2", "geometry"]],
    bk_circles[["Gattung lang", "Baumhoehe", "Kronendurchmesser", "geometry"]],
    how="inner",
    predicate="intersects",
)

# Many BK trees may hit one polygon — keep nearest-centroid match per pipeline polygon
tree_centroids = tree_gdf.set_index("tree_id").geometry.centroid
joined["dist"] = joined.apply(
    lambda r: tree_centroids[r["tree_id"]].distance(
        bk_circles.loc[r["index_right"]].geometry.centroid
    ), axis=1
)
matched = joined.sort_values("dist").groupby("tree_id", sort=False).first().reset_index()
matched["error_m"]   = matched["height_m"] - matched["Baumhoehe"]
matched["abs_err_m"] = matched["error_m"].abs()

print(f"Pipeline tree polygons : {len(tree_gdf)}")
print(f"Matched to BK tree     : {len(matched)}  "
      f"({len(matched)/len(tree_gdf)*100:.0f}% coverage)")
print(f"\nOverall  bias={matched['error_m'].mean():+.1f} m  "
      f"MAE={matched['abs_err_m'].mean():.1f} m  "
      f"RMSE={((matched['error_m']**2).mean())**0.5:.1f} m\n")

genus_col = matched["Gattung lang"].str.split(",").str[0]
stats = matched.groupby(genus_col).agg(
    n       =("error_m", "count"),
    bias_m  =("error_m", "mean"),
    mae_m   =("abs_err_m", "mean"),
).sort_values("n", ascending=False)
print(stats.head(12).to_string())

# --- Figures ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(matched["Baumhoehe"], matched["height_m"], alpha=0.4, s=15, c="steelblue")
lim = [0, max(matched[["Baumhoehe", "height_m"]].max()) * 1.05]
ax.plot(lim, lim, "k--", linewidth=1, label="1:1 line")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Baumkataster height (m)")
ax.set_ylabel("Pipeline estimate (m)")
ax.set_title("Estimated vs measured height")
ax.legend()

ax = axes[1]
matched.assign(genus=genus_col).boxplot(
    column="error_m", by="genus", ax=ax, rot=30, grid=False
)
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Genus")
ax.set_ylabel("Error (estimated − measured, m)")
ax.set_title("Height error by species")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Per-genus allometric profiles

Fit `ln(H) = A + B·ln(CPA)` separately for each genus with at least 200 valid trees.
Copy the printed dict into `ALLOMETRIC_PROFILES` in `src/config.py`.

In [ ]:
N_MIN = 200   # minimum trees per genus to fit a profile

cal_all = bk[["Gattung lang", "Baumhoehe", "Kronendurchmesser"]].copy()
cal_all = cal_all[(cal_all["Baumhoehe"] > 1) & (cal_all["Kronendurchmesser"] > 0.5)]
cal_all["genus"]  = cal_all["Gattung lang"].str.split(",").str[0]
cal_all["CPA_m2"] = np.pi * (cal_all["Kronendurchmesser"] / 2) ** 2
cal_all["ln_H"]   = np.log(cal_all["Baumhoehe"])
cal_all["ln_CPA"] = np.log(cal_all["CPA_m2"])

profiles = {}
for genus, grp in cal_all.groupby("genus"):
    if len(grp) < N_MIN:
        continue
    X = np.column_stack([np.ones(len(grp)), grp["ln_CPA"].values])
    B_hat, *_ = np.linalg.lstsq(X, grp["ln_H"].values, rcond=None)
    A_g, B_g = float(B_hat[0]), float(B_hat[1])
    pred = A_g + B_g * grp["ln_CPA"].values
    r2 = 1 - np.sum((grp["ln_H"].values - pred) ** 2) / \
              np.sum((grp["ln_H"].values - grp["ln_H"].mean()) ** 2)
    profiles[genus] = (round(A_g, 4), round(B_g, 4), round(r2, 3), len(grp))

# Print table
print(f"{'Genus':<18} {'n':>6}  {'A':>7}  {'B':>7}  {'R²':>6}")
print("-" * 52)
for genus, (A_g, B_g, r2, n) in sorted(profiles.items(), key=lambda x: -x[1][3]):
    print(f"{genus:<18} {n:>6}  {A_g:>7.4f}  {B_g:>7.4f}  {r2:>6.3f}")

# Print paste-ready dict for src/config.py
print("\n# ── paste into ALLOMETRIC_PROFILES in src/config.py ──")
print("ALLOMETRIC_PROFILES: dict[str, tuple[float, float]] = {")
for genus, (A_g, B_g, r2, n) in sorted(profiles.items(), key=lambda x: -x[1][3]):
    print(f'    "{genus}": ({A_g}, {B_g}),  # R²={r2}, n={n}')
print("}")

## Per-genus crown radius thresholds (watershed)

Median `Kronendurchmesser / 2` per genus — used as the species-aware watershed split
threshold in `vectorize_trees`. Copy the printed dict into `CROWN_RADIUS_BY_GENUS`
in `src/config.py`.

In [ ]:
cr_data = bk[["Gattung lang", "Kronendurchmesser"]].copy()
cr_data = cr_data[cr_data["Kronendurchmesser"] > 0.5]
cr_data["genus"]     = cr_data["Gattung lang"].str.split(",").str[0]
cr_data["crown_r_m"] = cr_data["Kronendurchmesser"] / 2

# Median crown radius and 95th-percentile per genus (n >= N_MIN)
cr_stats = (
    cr_data.groupby("genus")["crown_r_m"]
    .agg(n="count", median="median", p95=lambda x: x.quantile(0.95))
    .query(f"n >= {N_MIN}")
    .sort_values("n", ascending=False)
)

print(f"{'Genus':<18} {'n':>6}  {'median r (m)':>13}  {'p95 r (m)':>10}")
print("-" * 54)
for genus, row in cr_stats.iterrows():
    print(f"{genus:<18} {row['n']:>6}  {row['median']:>13.2f}  {row['p95']:>10.2f}")

# Print paste-ready dict for src/config.py
# Use 95th-percentile as threshold so most single trees still pass unsplit.
print("\n# ── paste into CROWN_RADIUS_BY_GENUS in src/config.py ──")
print("CROWN_RADIUS_BY_GENUS: dict[str, float] = {")
for genus, row in cr_stats.iterrows():
    print(f'    "{genus}": {round(row["p95"], 1)},  # median={row["median"]:.1f} m, n={row["n"]}')
print("}")